In [ ]:
import numpy as np, random, re
import nltk
nltk.download('punkt'); nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
STOP_WORDS=set(stopwords.words('english'))
def preprocess(text):
    text=text.lower();text=re.sub(r'[^a-zA-Z\s]',' ',text);words=word_tokenize(text)
    return [w for w in words if w not in STOP_WORDS and len(w)>2]
docs=["Machine learning AI", "Deep neural networks", "Natural language text", "Computer vision images", "Reinforcement agents"]
processed=[preprocess(d) for d in docs]
vocab={w:i for i,w in enumerate(set(w for d in processed for w in d))}
doc_ids=[[vocab[w] for w in d] for d in processed]
class LDA:
    def __init__(self, K=2, iters=50): self.K, self.iters = K, iters
    def fit(self, docs, vocab):
        D, V = len(docs), len(vocab)
        self.id2word={i:w for w,i in vocab.items()}
        self.dt=np.zeros((D,self.K));self.tw=np.zeros((self.K,V));self.tz=np.zeros(self.K))
        self.assign=[[] for _ in range(D)]
        for d,doc in enumerate(docs):
            for w in doc:
                k=random.randint(0,self.K-1);self.assign[d].append(k);self.dt[d][k]+=1;self.tw[k][w]+=1;self.tz[k]+=1
        for _ in range(self.iters):
            for d,doc in enumerate(docs):
                for i,w in enumerate(doc):
                    k=self.assign[d][i];self.dt[d][k]-=1;self.tw[k][w]-=1;self.tz[k]-=1
                    p=[(self.tw[k][w]+0.01)/(self.tz[k]+V*0.01)*(self.dt[d][k]+0.1) for k in range(self.K)]
                    p=np.array(p)/sum(p);new_k=np.random.choice(self.K,p=p)
                    self.assign[d][i]=new_k;self.dt[d][new_k]+=1;self.tw[new_k][w]+=1;self.tz[new_k]+=1
    def show(self,n=5):
        for k in range(self.K):print(f"Topic {k+1}: {' '.join([self.id2word[i] for i in self.tw[k].argsort()[-n:][::-1]})
lda=LDA();lda.fit(doc_ids,vocab);lda.show()